# 自回归生成与解码

> 上一部分讲完了训练。训练之后，模型学会了一件事：给词表里每个 Token 打分。但是打分不是终点。真正生成文字的时候，模型每预测一个词，只会交出一组 logits。这组 logits 还不是文字。要从几万个候选里选出一个，才算写出下一个 Token。
>
> 这一章只回答一个问题：**分数怎么变成 Token？**
>
> 本章介绍解码策略的四组核心内容。
>
> 1. **从 logits 到概率**：softmax 把任意分数变成合法的概率分布。这是所有选择规则的地基。
> 2. **三种基本策略**：Greedy 永远选最高分。Temperature 调整分布形状。Top-k / Top-p 裁掉长尾候选。
> 3. **两个常见补丁**：Repetition Penalty 治复读。Beam Search 同时考虑多条路径。
> 4. **完整拼装**：这些规则在一次真实采样里按什么顺序生效。
>
> 读完后，你应该能看懂 Hugging Face、vLLM、SGLang 文档里的 `temperature`、`top_p`、`top_k`、`repetition_penalty` 各自在改什么。它们改变的都是「怎么选」。它们没有改模型本身。

先把 logits 具体化。假设词表只有 6 个候选。模型刚读完「法国的首都是」。它输出了下面这组分数：

```text
巴黎   4.2
伦敦   3.8
北京   1.2
东京   0.8
香蕉  -0.5
。     0.4
```

看起来直接选「巴黎」就可以了。但事情没这么简单。有时候，你希望模型别每次都给相同的回答。有时候，模型会陷入复读。还有些时候，反而选择第二名的分数效果更好。这些需求会一个个变成具体的选择规则。先从最基本的问题开始：logits 到底怎么读？

## 1. 从 Logits 到概率

logits 是什么？说简单点，就是模型给每个候选打的原始分。

这个分还不能直接用。为什么？看上面那张表。你能发现两个特点。

第一个特点：全部加起来不等于 1。概率加起来要等于 1。这些分数加起来不是 1。

第二个特点：分数可能是负数。你看「香蕉」，是 -0.5。负数是模型觉得这个词非常不合适。

所以 logits 只能比大小，不能当概率来读。

那怎么把分数变成概率？靠 softmax。

softmax 的做法很简单，分两步。第一步，给每个分数取指数。第二步，除以所有指数的总和。

取指数有什么用？两个作用。一是把负数也变成正数。二是把分数之间的差距放大。

举个例子。「巴黎」和「伦敦」的 logit 只差 0.4。softmax 之后，「巴黎」的概率是「伦敦」的 $e^{0.4} \approx 1.5$ 倍。再看「香蕉」和「。」这两个候选。「香蕉」看似只比「。」低 0.9 分。概率上呢？「香蕉」几乎出局。

In [ ]:
import torch
import torch.nn.functional as F

tokens = ["巴黎", "伦敦", "北京", "东京", "香蕉", "。"]
logits = torch.tensor([4.2, 3.8, 1.2, 0.8, -0.5, 0.4])

probs = F.softmax(logits, dim=-1)
for t, l, p in zip(tokens, logits, probs):
    print(f"{t:>4}  logit={l:>4.1f}  p={p.item():.3f}")

print()
print(f"关键观察：巴黎/伦敦 logit 只差 0.4，概率比值却是 "
      f"{float(probs[0] / probs[1]):.2f} 倍（= e^0.4）")
print(f"香蕉 logit=-0.5 看着不大，概率只剩 {float(probs[4]):.4f}——指数放大了差距")

这里有两个细节。我们停一下。

第一个细节是什么？「香蕉」的 -0.5 和「。」的 0.4 看起来只差不到 1 分。概率上却是 9 倍的差距。这说明 logit 的「分差」和概率的「倍差」不是一回事。

第二个细节呢？概率加起来正好是 1。从这一步开始，选 Token 就变成按概率随机选一个。怎么随机选？这就是接下来所有策略要回答的问题。

## 2. Greedy 解码

greedy 是什么？说简单点，就是每次都选概率最高的 Token。`argmax` 一步到位。

它有两个优点。第一个优点，结果稳定。同样的输入，永远得到同样的输出。这样方便复现，也方便调试。第二个优点，计算最省。没有任何额外步骤。

什么时候用 greedy？闭卷问答、代码补全这类场景。这类场景要的就是那个正确答案。greedy 往往就是默认配置。

但稳定也意味着单调。怎么理解？比如让模型写一个关于春天的开头。greedy 每次都给出同一句。哪怕这句平平无奇。概率排第二的候选，可能也很合理。但它永远没机会被选到。

带着这个遗憾，我们先看第一种改进。改进发生在采样之前：先调整概率本身的形状。

In [ ]:
next_id = torch.argmax(logits).item()
print("Greedy 选择:", tokens[next_id])

print()
print("关键观察：argmax 是确定性的——同样的 logits，重复一万次也永远选同一个 Token")
print("第二名「伦敦」哪怕只差 0.4 分，也永远没有出场机会")

## 3. Temperature 与分布形状

Temperature 只做一件事。在 softmax 之前，把所有 logits 除以一个数 $T$。

公式长这样：

$$
p_i = \mathrm{softmax}(z_i / T)
$$

这个除法有什么用？看 $T$ 的大小。

如果 $T$ 小于 1，会发生什么？除以一个小于 1 的数，等于放大分数差距。分布会变「尖」。模型会更笃定。行为接近 greedy。

如果 $T$ 大于 1，会发生什么？除以一个大于 1 的数，等于压平差距。分布会变「平」。低分候选也分到概率。输出更多样。

有一句话值得先说在前面：**Temperature 不会删除任何候选。** 无论温度多高，「香蕉」的概率都不会变成零。只是每个候选的概率被重新分配了大小。这一点马上会引出新的问题。

In [ ]:
for T in [0.2, 0.7, 1.0, 1.5]:
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T:<3}: " + ", ".join(f"{t}:{x:.2f}" for t, x in zip(tokens, p.tolist())))

print()
print("关键观察：T=0.2 时前两名接近垄断；T=1.5 时候选差距被压平")
print("但无论 T 多大，香蕉的概率都不为 0——高温把长尾放出来了")

## 4. Top-k 与 Top-p 截断

Temperature 调高之后，一个副作用出现了。原本概率接近零的候选，也分到了不小的概率。一次随机选择，可能选到「香蕉」开头的句子。这样一来，整段生成就毁了。

我们想达到什么效果？让「合理的候选」机会更均等。同时，把「明显不合理」的候选挡在门外。这就是截断要做的事。

截断有几种做法？两种。先看 Top-k。

**Top-k** 最简单。只保留分数最高的 k 个候选。其余全部出局。实现上，把它们的 logit 设成 $-\infty$。这样 softmax 之后，它们的概率就是零。举例：k=2，就是只在巴黎、伦敦两个里选。

Top-k 有一个死板之处。什么死板？不同 Prompt 的分布形状差别很大。我们看两种情况。

第一种，模型很确定。前两名可能就占了 95% 的概率。这时候 k=5 会怎样？会把不靠谱的候选放进来了。

第二种，模型很犹豫。前 20 名概率都差不多。这时候 k=5 会怎样？又裁得太狠，把可能合理的候选漏掉了。

怎么解决？用 **Top-p**。Top-p 也叫 Nucleus Sampling。

Top-p 不看候选的个数。它看概率的质量。怎么做？把候选按概率从高到低累加。保留累计概率刚好盖住 p 的最小集合。

我们走一遍累加过程。假设概率是这样的：

```text
A 0.50   累计 0.50
B 0.25   累计 0.75
C 0.15   累计 0.90  ← 盖住 0.90，到此为止
D 0.06   累计 0.96
...
```

现在 `top_p = 0.9`。保留谁？保留 A、B、C 三个。D 和它后面的全部出局。

为什么会说 Top-p 是「动态」的？因为分布尖的时候，它自动少留几个。分布平的时候，它自动多留几个。

In [ ]:
def top_k_filter(logits, k):
    if k is None or k >= logits.numel():
        return logits
    threshold = torch.topk(logits, k).values[-1]
    return torch.where(logits < threshold, torch.tensor(float("-inf")), logits)

def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out[sorted_idx] = sorted_logits
    return out

cases = [
    ("top_k=2", top_k_filter(logits.clone(), 2)),
    ("top_p=0.9", top_p_filter(logits.clone(), 0.9)),
]
for name, filtered in cases:
    p = F.softmax(filtered, dim=-1)
    kept = [(t, round(x, 3)) for t, x in zip(tokens, p.tolist()) if x > 0]
    print(name, kept)

print()
print("关键观察：top_k=2 只剩巴黎伦敦；top_p=0.9 留到累计概率盖住 0.9 为止")

## 5. Repetition Penalty

前面所有策略有一个共同的盲区。它们只看当前这一步的概率。它们不知道前面已经生成过什么。

这个盲区会带来一种经典失败。比如模型生成：

```text
这家餐厅的菜非常好，非常好，非常好，非常好……
```

为什么会这样？因为「非常好」成了高分候选。greedy 会永远选它。sampling 也可能连续选中它。

这是模型的错吗？不是。问题在选择规则。选择规则里，没有任何机制惩罚「刚说过的词」。

怎么办？用 Repetition Penalty。它补上这个机制。对已经出现过的 Token，把它的 logit 压低一点。

Hugging Face 的实现规则是什么？正 logit 除以惩罚系数 $c$。负 logit 乘以 $c$。注意 $c > 1$。而且只作用于出现过的 Token。

公式如下：

$$
z_i' = \begin{cases} z_i / c & z_i > 0 \\ z_i \cdot c & z_i \le 0 \end{cases}
$$

我们手算一个关键情形。「巴黎」已经出现过一次。取 $c = 1.3$。它的 logit 是 4.2。压成多少？4.2 / 1.3 ≈ 3.23。没出现过的「伦敦」保持 3.8。压完之后，第一名和第二名交换。下一次选择自然翻转。

penalty 不删除候选。它只是让已经出现过的 Token 暂时分数变低。

In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty):
    """压低已出现 token 的 logit：正数除以 penalty，负数乘以 penalty"""
    z = logits.clone()
    for i in set(generated_ids):
        if z[i] > 0:
            z[i] = z[i] / penalty
        else:
            z[i] = z[i] * penalty
    return z

generated_ids = [tokens.index("巴黎")]  # 「巴黎」已经出现过一次
z_new = apply_repetition_penalty(logits, generated_ids, penalty=1.3)

for i, (t, old, new) in enumerate(zip(tokens, logits, z_new)):
    mark = "  <- 已出现，被压低" if i in generated_ids else ""
    print(f"{t:>3}  原始 {old:>5.2f}  压后 {new:>5.2f}{mark}")

print()
print("关键观察：argmax 从「", tokens[logits.argmax()], "」变成「", tokens[z_new.argmax()], "」")

In [ ]:
# 图里看更直观：只有出现过的 Token 被压低
import matplotlib.pyplot as plt

labels = ["Paris", "London", "Beijing", "Tokyo", "banana", "."]
xs = range(len(labels))
width = 0.38

plt.figure(figsize=(7, 3.5))
plt.bar([i - width / 2 for i in xs], logits.tolist(), width=width,
        label="original logits")
plt.bar([i + width / 2 for i in xs], z_new.tolist(), width=width,
        label="after repetition penalty")
plt.xticks(list(xs), labels)
plt.ylabel("logit")
plt.title("Only the repeated token (Paris) gets pushed down")
plt.legend()
plt.show()

## 6. Beam Search

到这里为止的所有策略，有一个共同假设。什么假设？每一步选局部最优。选好之后，整条输出就不会差。真的如此吗？我们看一个只有两步的例子。

```text
Step 1:  A: 0.6    B: 0.4
Step 2:  A 之后: x: 0.5 / y: 0.5
         B 之后: x: 0.9 / y: 0.1
```

Greedy 会怎么走？第一步选 A，因为 0.6 比 0.4 高。之后无论怎么走，整条结果的概率最多是 0.6 × 0.5 = 0.30。

那如果第一步选 B 呢？B 后面接 x，整条结果的概率是 0.4 × 0.9 = 0.36。**第一步分数排第二的候选，整条结果反而更好。**

这说明什么？局部最优不等于全局最优。这就是 Beam Search 存在的理由。

Beam Search 怎么做？每一步不再只保留 1 个候选。它同时保留 k 条路径。这里的 k 叫 beam width。它用整条候选序列的累计得分互相比较。最后输出总分最高的那一条。

工程实现里，累计得分怎么算？用 log 概率相加，而不是概率连乘。为什么？一连串小数相乘会下溢。取对数变成加法，就稳定了。这里数字小，直接乘就能看清楚。

那为什么现在的聊天模型几乎不用它？因为 Beam Search 有一个假设。它假设「整条结果概率最高的输出，就是最好的输出」。这个假设在翻译、摘要这类任务上通常成立。这些任务有标准答案。但开放式聊天不一样。开放式聊天里，概率最高的回答往往是「最平、最安全」的那个。多样性反而没了。所以现代对话生成默认用 sampling。beam search 主要留在封闭任务里。

In [ ]:
# 用同一组数字验证：第一步的第一名，不一定是整条路的赢家
step1 = {"A": 0.6, "B": 0.4}
step2 = {"A": {"x": 0.5, "y": 0.5}, "B": {"x": 0.9, "y": 0.1}}

greedy_first = max(step1, key=step1.get)
greedy_total = step1[greedy_first] * max(step2[greedy_first].values())
print(f"Greedy 第一步选 {greedy_first}，之后接着走局部最优，总分 {greedy_total:.2f}")

all_paths = {(t1, t2): p1 * p2
             for t1, p1 in step1.items()
             for t2, p2 in step2[t1].items()}
best_path = max(all_paths, key=all_paths.get)
print("所有完整路径:", all_paths)
print("关键观察：整条路的最优是", "".join(best_path),
      "总分", all_paths[best_path])

In [ ]:
# 把这棵两步搜索树画出来：线越粗概率越高，绿色是 Beam 找到的整条最优路径
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
nodes = {"start": (0, 0.5), "A": (1, 0.78), "B": (1, 0.22),
         "Ax": (2, 0.95), "Ay": (2, 0.62), "Bx": (2, 0.38), "By": (2, 0.05)}
edges = [("start", "A", 0.6), ("start", "B", 0.4),
         ("A", "Ax", 0.5), ("A", "Ay", 0.5),
         ("B", "Bx", 0.9), ("B", "By", 0.1)]

for a, b, p in edges:
    (x1, y1), (x2, y2) = nodes[a], nodes[b]
    on_best = (a, b) in {("start", "B"), ("B", "Bx")}
    ax.plot([x1, x2], [y1, y2], color="tab:green" if on_best else "tab:gray",
            linewidth=1 + 8 * p, alpha=1.0 if on_best else 0.45, zorder=1)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.04, f"{p:.1f}",
            ha="center", fontsize=9)

for name, (x, y) in nodes.items():
    ax.text(x, y, name, ha="center", va="center", zorder=2,
            bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black"))

ax.text(*nodes["Ax"], "  A-x total 0.30\n  (greedy's choice)", va="center", fontsize=9)
ax.text(*nodes["Bx"], "  B-x total 0.36\n  (best, beam finds it)", va="center", fontsize=9)
ax.set_xlim(-0.2, 3.6)
ax.set_ylim(-0.1, 1.15)
ax.axis("off")
ax.set_title("Beam width 2: the best full path may start with the runner-up")
plt.show()

## 7. 一次完整的采样流程

把前面的规则按真实框架的处理顺序串起来，一次 Decode step 大致长这样：

```text
model forward
    ↓
logits
    ↓ repetition / presence / frequency penalties（先处理「历史」）
temperature（再调整形状）
    ↓
top-k / top-p / min-p（最后裁剪）
    ↓
sampling（multinomial 抽签）
    ↓
next token
```

这个顺序有意义吗？有意义。

为什么 penalty 在最前面？因为 penalty 依赖「已经生成过什么」。它必须最先改分数。

temperature 和截断呢？它们都在「改分布」。先后影响不大。但它们都发生在采样之前。

不同框架的 processor 顺序可能有差异。生产环境以具体实现为准。

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    x = logits.clone() / max(temperature, 1e-5)
    x = top_k_filter(x, top_k)
    if top_p is not None:
        x = top_p_filter(x, top_p)
    probs = F.softmax(x, dim=-1)
    return torch.multinomial(probs, 1).item(), probs

print("greedy:", tokens[torch.argmax(logits).item()])
for name, cfg in [
    ("T=0.7, p=0.9", dict(temperature=0.7, top_p=0.9)),
    ("T=1.2, p=0.95", dict(temperature=1.2, top_p=0.95)),
]:
    picks = [tokens[sample_next(logits, seed=s, **cfg)[0]] for s in range(8)]
    print(name, picks)


## 8. 常用生成参数

把这一章出现过的参数放进一张表。以后在 API 文档、模型卡、招聘 JD 里看到它们，先问一句：它作用在哪一层？是改分数？还是改形状？还是裁候选？还是管停止条件？

| 参数 | 作用 |
|:---|:---|
| `temperature` | 调整概率分布的形状（尖或平） |
| `top_k` | 只保留分数最高的 k 个候选 |
| `top_p` | 按累计概率质量动态截断 |
| `repetition_penalty` | 压低已出现 Token 的分数 |
| `max_tokens` / `max_new_tokens` | 生成长度上限 |
| `stop` / EOS | 停止条件 |
| `seed` | 采样随机性的种子 |

## 小结

这一章从一组 logits 出发。我们把「怎么选 Token」拆成了一层层规则。

- softmax 把分数变成概率。这是一切规则的起点。
- Greedy 稳定但单调。Temperature 不删候选。它只重新分配概率。
- Top-k 数候选个数。Top-p 数概率质量。它们都在解决「高温后长尾混进来」。
- Repetition Penalty 把出现过的 Token 压低。它治复读。
- Beam Search 同时保留 k 条路径找全局最优。开放式聊天更常用 sampling。
- 这些全是「选 Token 的规则」。从头到尾，模型本身一个参数都没动。

参数没动。下一章的问题也随之而来：

> 为了得到这组 logits，模型一次前向到底做了什么？为什么这么慢？

## 作业

三道题分别对应三类选择规则：penalty、截断、Temperature。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤。但不建议直接让 AI 完成题目。
> 这些规则都不长。自己写一遍。这样印象最深。

### 作业 1：实现 repetition penalty

只压低出现过的 Token。正 logit 除以 penalty。负 logit 乘以 penalty。

**小提示**：遍历 `generated_ids` 里的下标。按 `z[i]` 的正负号决定做除法还是乘法。

In [ ]:
# 作业 1：repetition penalty 填空

test_logits = torch.tensor([4.2, 3.8, -0.5, 0.4])
test_generated = [0]  # 下标 0 的 token 已经出现过

def penalize(logits, generated_ids, penalty):
    """返回压低已出现 token 之后的新 logits"""
    z = logits.clone()
    for i in set(generated_ids):
        # TODO：把下面三引号里的内容替换成你的代码
        """在这里按正负号对 z[i] 做除法或乘法"""
    return z

result = penalize(test_logits, test_generated, 1.2)
assert torch.isclose(result[0], torch.tensor(4.2 / 1.2)), result
assert torch.isclose(result[1], torch.tensor(3.8)), result
print("✅ 作业 1 通过：你实现了复读抑制的基本规则")

### 作业 2：算出 top-p 保留几个候选

概率已按从高到低排序。top-p 要保留「累计概率刚好盖住 p」的最小候选集合。

**小提示**：从高到低累加。找到第一个让累计和大于等于 p 的位置。保留到这个位置为止。

In [ ]:
# 作业 2：top-p 候选数 填空

probs = torch.tensor([0.50, 0.25, 0.15, 0.06, 0.04])  # 已按从高到低排序

def top_p_keep_count(probs, p):
    """返回 top-p 截断后保留的候选个数"""
    # TODO：把下面三引号里的内容替换成你的代码
    """在这里用累计概率算出需要保留几个候选"""

assert top_p_keep_count(probs, 0.5) == 1
assert top_p_keep_count(probs, 0.9) == 3
assert top_p_keep_count(probs, 0.99) == 5
print("✅ 作业 2 通过：你理解了 top-p 是按概率质量动态截断")

### 作业 3：用熵量化 Temperature 的效果

分布越「平」，采样结果越难预测。熵（entropy）就是量化这件事的标准指标。

**小提示**：先算 `F.softmax(logits / T, dim=-1)`。熵的定义是 $-\sum_i p_i \log p_i$。

In [ ]:
# 作业 3：不同 Temperature 下的熵 填空

def entropy(probs):
    """计算分布的熵：越大表示分布越平，采样越难预测"""
    p = probs[probs > 0]
    return float(-(p * p.log()).sum())

def probs_at_temperature(T):
    # TODO：把下面三引号里的内容替换成你的代码
    """返回 logits 除以 T 之后 softmax 的概率分布"""

assert entropy(probs_at_temperature(0.3)) < entropy(probs_at_temperature(1.0))
assert entropy(probs_at_temperature(1.0)) < entropy(probs_at_temperature(3.0))
print("✅ 作业 3 通过：低温分布更尖、高温分布更平，你能量化它了")